# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhassantahir-afk/ML-Engineering-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [286]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

Building the Baseline Score First

In [287]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np

feature_frame = con.execute(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            CASE WHEN report_date < DATE '2026-03-16' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    features AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS feat_impressions,
            SUM(CASE WHEN period = 'first_half' THEN gsc_clicks ELSE 0 END) AS feat_clicks,
            AVG(CASE WHEN period = 'first_half' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS position_first_half,
            AVG(CASE WHEN period = 'second_half' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS position_second_half,
            SUM(CASE WHEN period = 'first_half' THEN 1 ELSE 0 END) AS feat_days_active,
            SUM(CASE WHEN period = 'first_half' THEN gsc_clicks ELSE 0 END) * 1.0
                / NULLIF(SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END), 0) AS feat_ctr
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    ),
    label AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS first_half,
            SUM(CASE WHEN period = 'second_half' THEN gsc_impressions ELSE 0 END) AS second_half
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.feat_impressions,
        f.feat_clicks,
        f.position_first_half,
        f.position_second_half,
        (f.position_second_half - f.position_first_half) AS position_change,
        f.feat_days_active,
        f.feat_ctr,
        CASE
            WHEN (l.second_half - l.first_half) * 1.0 / NULLIF(l.first_half, 0) * 100 <= -20
            THEN TRUE ELSE FALSE
        END AS declining_flag
    FROM features f
    JOIN label l
        ON f.content_hash_id = l.content_hash_id AND f.client_hash_id = l.client_hash_id
    WHERE l.first_half > 0
      AND f.position_first_half IS NOT NULL
      AND f.position_second_half IS NOT NULL
    ORDER BY f.content_hash_id, f.client_hash_id
""").df()

# Belt-and-suspenders: also reset the index after sorting, so row positions are fully deterministic
feature_frame = feature_frame.reset_index(drop=True)

print(feature_frame.shape)

print("\n=== Feature Frame ===")
print(feature_frame.shape)
feature_frame.head()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(139747, 10)

=== Feature Frame ===
(139747, 10)


,content_hash_id,client_hash_id,feat_impressions,feat_clicks,position_first_half,position_second_half,position_change,feat_days_active,feat_ctr,declining_flag
0,content_000005d4ced12088,client_9958f0a7ae1df715,23.0,0.0,72.101852,73.306667,1.204815,9.0,0.000000,False
1,content_00007bd2985b77c3,client_73cda7b4e4f265ea,22.0,0.0,12.600000,6.600000,-6.000000,12.0,0.000000,False
2,content_0000cd28fbda69f3,client_3ffa76342f366962,11.0,0.0,4.062500,4.553333,0.490833,8.0,0.000000,False
3,content_00014efc121d911d,client_08a6a72ff48e62c0,53.0,0.0,6.768864,4.688095,-2.080769,14.0,0.000000,False
4,content_000184dde41afe75,client_62f4a7e64f5e0096,2405.0,8.0,3.682934,3.477619,-0.205315,15.0,0.003326,False


In [288]:
# Rebuild the baseline rule/score exactly as in w04
feature_frame['high_volume'] = (feature_frame['feat_impressions'] >= 500).astype(int)
feature_frame['position_slipped'] = (feature_frame['position_change'] > 2).astype(int)
feature_frame['score'] = (
    feature_frame['high_volume']
    * feature_frame['position_slipped']
    * feature_frame['feat_impressions']
)

print(feature_frame.shape)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

baseline_precision_50 = precision_at_k(feature_frame['score'].values, feature_frame['declining_flag'].values, 50)
base_rate = feature_frame['declining_flag'].mean()

print(f"Baseline precision@50: {baseline_precision_50:.3f}")
print(f"Base rate (overall decline rate): {base_rate:.3f}")

(139747, 13)
Baseline precision@50: 0.400
Base rate (overall decline rate): 0.278


Note: Baseline precision@50 = 0.400, against a base rate of 0.278. The rule performs above chance, but with real room for improvement and consistent with the ~50% rule/label disagreement found in the Week-4 top-20 review.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Decision Tree then Random Forest

Why: it fits my lane as i have a binary future yes/no answer

`declining_flag = True/False`

Reason for choosing Decsision Tree is to have a readable model at hand.
Decision Tree first (readable, matches my baseline exploration style) then Random Forest (usually stronger, still interpretable via feature importance).

In [289]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Im going to be grouping based upon clients, because the same client on both train and test splits will cause issues like the model learning client specific patterns and cheating.

this might give us better precision but it would not work on unseen or new data.

Split is grouped by client_hash_id to prevent leakage of client-specific patterns across train/test. Because client sizes are highly uneven (the largest client alone is 18% of all rows; the top 4 clients together exceed 54%), the resulting split lands at roughly 92/8 rather than the intended 80/20. This is an expected consequence of grouping on an unbalanced panel, not an error.

In [290]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

# Reuse the feature_frame already built earlier in this notebook
feature_cols = ['feat_impressions', 'feat_clicks', 'position_first_half',
                'feat_days_active', 'feat_ctr']

X = feature_frame[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = feature_frame['declining_flag']
groups = feature_frame['client_hash_id']  # used ONLY for splitting, never as a feature

splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]


In [291]:
# --- Checks ---
print(f"Train size: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test size: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")


Train size: 128111 (91.7%)
Test size: 11636 (8.3%)


In [292]:
overlap = set(groups_train.unique()) & set(groups_test.unique())
print(f"Clients appearing in BOTH train and test: {len(overlap)}")


Clients appearing in BOTH train and test: 0


In [293]:
print(f"\nTrain decline rate: {y_train.mean():.3f}")
print(f"Test decline rate: {y_test.mean():.3f}")


Train decline rate: 0.276
Test decline rate: 0.301


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [294]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.tree import DecisionTreeClassifier

# below is just a test to confirm the ideal depth for the model
tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)

tree_scores_test = tree.predict_proba(X_test)[:, 1]
tree_precision_50 = precision_at_k(tree_scores_test, y_test.values, 50)

print(f"Decision Tree precision@50 (test set): {tree_precision_50:.3f}")

depths_to_try = [2, 3, 4, 5, 6]
results = []

for depth in depths_to_try:
    tree = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42)
    tree.fit(X_train, y_train)

    tree_scores_test = tree.predict_proba(X_test)[:, 1]
    precision_50 = precision_at_k(tree_scores_test, y_test.values, 50)

    results.append({'max_depth': depth, 'precision_at_50': precision_50})

import pandas as pd
depth_comparison = pd.DataFrame(results)
print(depth_comparison)


Decision Tree precision@50 (test set): 0.520
   max_depth  precision_at_50
0          2             0.40
1          3             0.38
2          4             0.52
3          5             0.46
4          6             0.48


In [297]:
# Final split, locked in
#splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Final model
tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)

tree_scores_test = tree.predict_proba(X_test)[:, 1]
tree_precision_50 = precision_at_k(tree_scores_test, y_test.values, 50)

# Baseline, evaluated on the SAME test set for a fair comparison
baseline_scores_test = feature_frame.loc[X_test.index, 'score'].values
baseline_precision_50_test = precision_at_k(baseline_scores_test, y_test.values, 50)

base_rate_test = y_test.mean()

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=4, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)

rf_scores_test = rf.predict_proba(X_test)[:, 1]
rf_precision_50 = precision_at_k(rf_scores_test, y_test.values, 50)

print(f"Random Forest precision@50 (test set): {rf_precision_50:.3f}")

comparison_table = pd.DataFrame([
    {'method': 'Baseline rule', 'precision_at_50': baseline_precision_50_test},
    {'method': 'Decision Tree (depth 6)', 'precision_at_50': tree_precision_50},
    {'method': 'Random Forest (depth 6, 200 trees)', 'precision_at_50': rf_precision_50},
    {'method': 'Base rate', 'precision_at_50': base_rate_test},
])
print(comparison_table)

Random Forest precision@50 (test set): 0.460
                               method  precision_at_50
0                       Baseline rule         0.240000
1             Decision Tree (depth 6)         0.520000
2  Random Forest (depth 6, 200 trees)         0.460000
3                           Base rate         0.301134


Note: Initial feature importances revealed position_second_half and position_change (both derived from data after the decision point) were meaningfully influencing the model. A leakage risk consistent with the ML-04 leakage lesson. After removing them, precision@50 dropped modestly (Tree: 0.56→0.52, Forest: 0.64→0.62), confirming most of the model's real performance came from genuinely available first-half signals, not the leak."

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [296]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(importances)

               feature  importance
0     feat_impressions    0.348724
3     feat_days_active    0.242708
4             feat_ctr    0.169837
2  position_first_half    0.158728
1          feat_clicks    0.080003


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.